# 🔊 XTTS-v2 음성 복제 TTS 서버 — Colab

동화 생성·이미지·감정 모델 없이 **한국어 XTTS-v2 음성 합성만** 실행하는 경량 서버입니다.
Google Drive와 기본 화자 파일 업로드가 필요 없습니다. XTTS 모델은 현재 Colab 런타임의 `/content`에만 저장됩니다.

## Flutter 호환 API

- `POST /api/tts`: `multipart/form-data`의 `text`와 `speaker_wav`(WAV)를 받습니다.
- Flutter에서 녹음한 `speaker_wav`를 기준 목소리로 사용해 해당 목소리로 읽습니다.
- 화자 WAV 없이 합성을 요청하면 서버는 `400`을 반환합니다.
- `POST /api/tts/warm-up`: 화자 WAV 없이 모델만 미리 GPU에 로드합니다.
- `GET /health`: 서버·모델 상태를 확인합니다.

**실행 순서:** 셀 1 → 셀 2 → 셀 3 → 셀 4.  
Coqui XTTS-v2 라이선스를 확인한 뒤 셀 2의 `XTTS_ACCEPT_LICENSE = True`로 변경하세요.

Flutter에서 보내는 녹음 파일은 합성 요청 중에만 `/tmp`에 저장되고 응답을 만든 직후 삭제됩니다. Colab 런타임이 종료되면 XTTS 캐시도 함께 사라집니다.


In [ ]:
# 셀 1: XTTS 전용 Python 환경과 서버 패키지 설치
!pip install -q -U huggingface_hub hf_transfer
!pip install -q fastapi "uvicorn[standard]" python-multipart nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import os, shutil, subprocess, sys, torch

# Colab Python 3.12에는 ensurepip이 빠진 경우가 있어 virtualenv를 사용합니다.
XTTS_VENV = '/content/xtts_env'
XTTS_PYTHON = f'{XTTS_VENV}/bin/python'
if not os.path.isfile(XTTS_PYTHON):
    shutil.rmtree(XTTS_VENV, ignore_errors=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'virtualenv'])
    subprocess.check_call([
        sys.executable, '-m', 'virtualenv', '--system-site-packages', XTTS_VENV
    ])

subprocess.check_call([XTTS_PYTHON, '-m', 'pip', 'install', '-q', '-U', 'pip'])
subprocess.check_call([
    XTTS_PYTHON, '-m', 'pip', 'install', '-q', '-U',
    'coqui-tts==0.27.5', 'fastapi', 'uvicorn[standard]', 'python-multipart'
])

cf = shutil.which('cloudflared')
if cf is None:
    for candidate in ['/usr/local/bin/cloudflared', '/usr/bin/cloudflared']:
        if os.path.exists(candidate):
            cf = candidate
            break
if cf is None:
    raise RuntimeError('cloudflared 설치에 실패했습니다. 셀을 다시 실행하세요.')

print('✅ XTTS 전용 환경 준비 완료')
print('XTTS Python:', XTTS_PYTHON)
print('cloudflared:', cf)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')


In [ ]:
# 셀 2: XTTS 런타임 설정 (화자 파일 업로드 불필요)
import os

# XTTS 모델 캐시는 현재 Colab 런타임에만 저장됩니다.
os.environ['TTS_HOME'] = '/content/xtts_cache'
os.makedirs(os.environ['TTS_HOME'], exist_ok=True)

# Coqui XTTS-v2 라이선스를 확인한 뒤 True로 바꾸세요.
XTTS_ACCEPT_LICENSE = False
if not XTTS_ACCEPT_LICENSE:
    print('⚠️ XTTS_ACCEPT_LICENSE = True로 바꾼 뒤 다음 셀을 실행하세요.')

os.environ['XTTS_ACCEPT_LICENSE'] = str(XTTS_ACCEPT_LICENSE).lower()
os.environ['XTTS_PYTHON'] = '/content/xtts_env/bin/python'
if XTTS_ACCEPT_LICENSE:
    os.environ['COQUI_TOS_AGREED'] = '1'

print('XTTS 캐시:', os.environ['TTS_HOME'])
print('화자 WAV: Flutter의 /api/tts 요청에서 speaker_wav로 받습니다.')


In [ ]:
%%writefile /content/xtts_voice_server.py
import asyncio
import base64
import os
import re
import threading
import wave
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from TTS.api import TTS

MODEL_NAME = 'tts_models/multilingual/multi-dataset/xtts_v2'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LICENSE_ACCEPTED = os.getenv('XTTS_ACCEPT_LICENSE', '').lower() == 'true'
MAX_TEXT_CHARS = 3000
MAX_SPEAKER_WAV_BYTES = 12 * 1024 * 1024

app = FastAPI(title='Fairytale XTTS-v2 API', version='1.0.0')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

_model = None
_model_lock = threading.Lock()


def clean_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def split_text(text: str, limit: int = 230) -> list[str]:
    sentences = re.split(r'(?<=[.!?。！？])\s*', clean_text(text))
    chunks, current = [], ''
    for sentence in filter(None, sentences):
        if len(sentence) > limit:
            if current:
                chunks.append(current)
                current = ''
            chunks.extend(sentence[index:index + limit] for index in range(0, len(sentence), limit))
        elif not current:
            current = sentence
        elif len(current) + len(sentence) + 1 <= limit:
            current = f'{current} {sentence}'
        else:
            chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks or [text]


def get_model():
    global _model
    if not LICENSE_ACCEPTED:
        raise RuntimeError('XTTS-v2 라이선스를 확인한 뒤 XTTS_ACCEPT_LICENSE=True로 설정하세요.')
    with _model_lock:
        if _model is None:
            print(f'[XTTS] {MODEL_NAME} 로딩 중: {DEVICE}')
            _model = TTS(model_name=MODEL_NAME, progress_bar=True).to(DEVICE)
            print('[XTTS] 준비 완료')
    return _model


def decode_speaker_wav(temp_dir: Path, speaker_wav_b64: str | None) -> Path:
    if not speaker_wav_b64:
        raise ValueError('speaker_wav가 필요합니다. Flutter에서 내 목소리를 먼저 녹음해 주세요.')
    try:
        raw = base64.b64decode(speaker_wav_b64, validate=True)
    except Exception as error:
        raise ValueError('speaker_wav는 올바른 Base64 WAV여야 합니다.') from error
    if not raw or len(raw) > MAX_SPEAKER_WAV_BYTES:
        raise ValueError('speaker_wav는 1바이트 이상 12MB 이하의 WAV여야 합니다.')
    candidate = temp_dir / 'speaker.wav'
    candidate.write_bytes(raw)
    try:
        with wave.open(str(candidate), 'rb') as audio:
            if audio.getnframes() <= 0:
                raise ValueError('speaker_wav에 음성 프레임이 없습니다.')
    except wave.Error as error:
        raise ValueError('speaker_wav는 PCM WAV 형식이어야 합니다.') from error
    return candidate


def synthesize(text: str, speaker_wav_b64: str | None = None) -> bytes:
    text = clean_text(text)
    if not text:
        raise ValueError('읽을 텍스트가 비어 있습니다.')
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f'텍스트는 {MAX_TEXT_CHARS}자 이하여야 합니다.')

    model = get_model()
    with TemporaryDirectory(prefix='xtts_request_') as temp:
        temp_dir = Path(temp)
        speaker_wav = decode_speaker_wav(temp_dir, speaker_wav_b64)
        parts = []
        for index, chunk in enumerate(split_text(text)):
            part = temp_dir / f'part_{index}.wav'
            model.tts_to_file(
                text=chunk,
                file_path=str(part),
                speaker_wav=str(speaker_wav),
                language='ko',
            )
            parts.append(part)

        output = BytesIO()
        with wave.open(output, 'wb') as merged:
            format_signature = None
            for part in parts:
                with wave.open(str(part), 'rb') as source:
                    signature = (
                        source.getnchannels(), source.getsampwidth(),
                        source.getframerate(), source.getcomptype(), source.getcompname(),
                    )
                    if format_signature is None:
                        format_signature = signature
                        merged.setnchannels(signature[0])
                        merged.setsampwidth(signature[1])
                        merged.setframerate(signature[2])
                        merged.setcomptype(signature[3], signature[4])
                    elif signature != format_signature:
                        raise RuntimeError('XTTS 청크의 WAV 형식이 서로 다릅니다.')
                    merged.writeframes(source.readframes(source.getnframes()))
        return output.getvalue()


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'engine': 'XTTS-v2',
        'device': DEVICE,
        'model_loaded': _model is not None,
        'speaker_wav_required': True,
        'speaker_wav_supported': True,
    }


@app.post('/api/tts/warm-up')
async def warm_up():
    try:
        await asyncio.to_thread(get_model)
        return {'status': 'ready', 'engine': 'XTTS-v2', 'device': DEVICE}
    except Exception as error:
        raise HTTPException(503, str(error)) from error


@app.post('/api/tts')
async def tts(request: Request):
    speaker_wav_b64 = None
    content_type = request.headers.get('content-type', '').lower()
    if content_type.startswith('multipart/form-data'):
        form = await request.form()
        text = str(form.get('text', ''))
        speaker_wav = form.get('speaker_wav')
        if speaker_wav is not None:
            if not hasattr(speaker_wav, 'read'):
                raise HTTPException(400, 'speaker_wav 형식이 올바르지 않습니다.')
            raw = await speaker_wav.read()
            if len(raw) > MAX_SPEAKER_WAV_BYTES:
                raise HTTPException(400, 'speaker_wav는 12MB 이하여야 합니다.')
            speaker_wav_b64 = base64.b64encode(raw).decode('ascii')
    else:
        try:
            body = await request.json()
        except Exception as error:
            raise HTTPException(400, 'JSON 또는 multipart 요청이 필요합니다.') from error
        text = str(body.get('text', ''))
        speaker_wav_b64 = str(body.get('speaker_wav_b64', '')).strip() or None

    try:
        audio = await asyncio.to_thread(synthesize, text, speaker_wav_b64)
        return Response(
            content=audio,
            media_type='audio/wav',
            headers={'Cache-Control': 'no-store'},
        )
    except ValueError as error:
        raise HTTPException(400, str(error)) from error
    except Exception as error:
        raise HTTPException(503, str(error)) from error


In [ ]:
# 셀 4: XTTS 서버 실행 + Cloudflare 터널
import nest_asyncio, os, re, shutil, subprocess, sys, threading, time, urllib.request
from pathlib import Path

nest_asyncio.apply()

if os.environ.get('XTTS_ACCEPT_LICENSE', '').lower() != 'true':
    raise RuntimeError('XTTS-v2 라이선스를 확인한 뒤 셀 2의 XTTS_ACCEPT_LICENSE = True로 바꾸세요.')
XTTS_PYTHON = os.environ.get('XTTS_PYTHON', '/content/xtts_env/bin/python')
if not Path(XTTS_PYTHON).is_file():
    raise RuntimeError('XTTS Python 환경을 찾지 못했습니다. 셀 1을 다시 실행하세요.')

def server_alive():
    try:
        urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=2)
        return True
    except Exception:
        return False

if not server_alive():
    log_path = '/tmp/xtts_voice_server.log'
    open(log_path, 'w').close()
    log_handle = open(log_path, 'a')
    subprocess.Popen(
        [XTTS_PYTHON, '-m', 'uvicorn', 'xtts_voice_server:app', '--host', '127.0.0.1', '--port', '8001'],
        cwd='/content',
        env=os.environ.copy(),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

for elapsed in range(90):
    if server_alive():
        break
    time.sleep(1)
else:
    details = Path('/tmp/xtts_voice_server.log').read_text(errors='replace')[-3000:]
    raise RuntimeError(f'XTTS 서버 시작 실패:\n{details}')

print('🔄 XTTS 모델 첫 로딩 중... 처음 한 번은 다운로드 때문에 수 분 걸릴 수 있습니다.')
warm_request = urllib.request.Request(
    'http://127.0.0.1:8001/api/tts/warm-up',
    data=b'{}',
    headers={'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urllib.request.urlopen(warm_request, timeout=1200) as response:
        print('✅ XTTS 준비 완료:', response.read().decode('utf-8'))
except Exception as error:
    details = Path('/tmp/xtts_voice_server.log').read_text(errors='replace')[-3000:]
    raise RuntimeError(f'XTTS 워밍업 실패: {error}\n{details}') from error

cf = shutil.which('cloudflared')
if not cf:
    raise RuntimeError('cloudflared를 찾지 못했습니다. 셀 1을 다시 실행하세요.')

cloudflared_log = '/tmp/xtts_cloudflared.log'
if os.path.exists(cloudflared_log):
    os.remove(cloudflared_log)
subprocess.Popen(
    [cf, 'tunnel', '--url', 'http://127.0.0.1:8001', '--logfile', cloudflared_log, '--loglevel', 'info'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

SERVER_URL = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(cloudflared_log):
        log = Path(cloudflared_log).read_text(errors='replace')
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', log)
        if match:
            SERVER_URL = match.group()
            break

if not SERVER_URL:
    raise RuntimeError('Cloudflare 터널 URL을 얻지 못했습니다.')

print('=' * 68)
print('🚀 XTTS 전용 서버 URL:', SERVER_URL)
print('🔊 TTS:              ', f'{SERVER_URL}/api/tts')
print('🔥 Warm-up:          ', f'{SERVER_URL}/api/tts/warm-up')
print('❤️ Health:           ', f'{SERVER_URL}/health')
print('=' * 68)
print('Flutter .env의 TTS_API_BASE_URL에 위 URL을 넣으세요.')


In [ ]:
# 셀 5: 서버 및 Flutter 녹음 연동 테스트 (선택사항)
import requests
from IPython.display import Audio, display

SERVER_URL = ''  # 셀 4 출력의 https://...trycloudflare.com 주소를 붙여넣으세요.
SPEAKER_WAV_PATH = ''  # 선택: Colab에 이미 있는 WAV 경로. 예: /content/my_voice.wav

if not SERVER_URL:
    print('SERVER_URL을 입력하세요.')
else:
    print('Health:', requests.get(f'{SERVER_URL}/health', timeout=30).json())
    warm = requests.post(f'{SERVER_URL}/api/tts/warm-up', json={}, timeout=1200)
    print('Warm-up:', warm.status_code, warm.text)

    if SPEAKER_WAV_PATH:
        with open(SPEAKER_WAV_PATH, 'rb') as wav:
            response = requests.post(
                f'{SERVER_URL}/api/tts',
                data={'text': '이 목소리로 따뜻한 동화를 읽어요.'},
                files={'speaker_wav': ('my_voice.wav', wav, 'audio/wav')},
                timeout=900,
            )
        print('TTS:', response.status_code, 'bytes:', len(response.content))
        if response.status_code == 200:
            display(Audio(data=response.content, rate=24000, autoplay=False))
        else:
            print(response.text)
    else:
        print('SPEAKER_WAV_PATH 없이도 모델 워밍업은 완료되었습니다.')
        print('Flutter가 speaker_wav를 보내면 /api/tts에서 바로 음성 합성이 됩니다.')
